# MOLDR: Decompose, Reassemble, and Train a Small Legacy Policy

Language: **English** | [日本語](quickstart_ja.ipynb)

This tutorial is a quick runnable walkthrough to verify the workflow end-to-end. Starting from 8 sample molecules, it covers **frequent substructure mining with `fast-gspan` -> inspecting reassembly candidates -> running one episode in the `legacy` environment -> saving results**. At the end, you can optionally run a short PPO training job with the same `legacy` configuration and sample molecules from the saved checkpoint.

By default, **Run All** runs on CPU only and does not start Ray, train a policy, or download any external dataset. It uses the bundled `models/model_300dim.pkl`. Only when you explicitly set `RUN_TRAINING = True` near the end will it run 1 iteration of PPO using local Ray on CPU. No W&B account is required.

**Matching implementation:** This notebook uses the shared `moldr.factory` / `moldr.cli` modules and the original `MolEnvValueMax` environment. Please run it with the same source revision, including the shared CLI. Because `legacy` and the **experimental** `E2E` mode differ in action spaces, observation spaces, and model architectures, checkpoints must always be restored alongside their saved configuration.

If you prefer running without Jupyter, see [the Python tutorial](../docs/tutorial.md), which also uses `legacy` defaults and walks through custom molecular scores, reward shaping, and sampling from saved checkpoints.

## 1. Environment and Output Directory

Follow the Installation instructions in the README and select this project's Python environment as the Jupyter kernel.
You can run the cells below either from the repository root or from `examples/`. First, print the Python executable path.

Each run saves its outputs into a new `outputs/tutorials/quickstart_.../` directory so original data and existing training runs are never overwritten.

In [ ]:
import io
import json
import sys
from contextlib import redirect_stdout
from pathlib import Path
from tempfile import mkdtemp

ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "moldr").is_dir() and (p / "pyproject.toml").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or examples/.")
if not (ROOT / "moldr/factory.py").is_file():
    raise RuntimeError("This notebook requires the matching development source tree.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from rdkit import Chem, rdBase
from rdkit.Chem import QED, Draw

from moldr.chemutils import get_mol, get_smiles, validate_reassembly_mol

model_path = ROOT / "models/model_300dim.pkl"
if not model_path.is_file():
    raise FileNotFoundError(f"Bundled Mol2Vec model not found: {model_path}")
output_parent = ROOT / "outputs/tutorials"
output_parent.mkdir(parents=True, exist_ok=True)
OUTPUT = Path(mkdtemp(prefix="quickstart_", dir=output_parent))
SEED = 7
print("Python:", sys.executable)
print("Repository:", ROOT)
print("Output:", OUTPUT)

## 2. Inspect the Sample Molecules

Here we use 8 small molecules chosen for illustration (not an excerpt from an experimental dataset).
The input format has a `smiles` column with one molecule per row. When replacing this with your own data, start with roughly this number of molecules first.

QED is used here as an example molecular score. Its values lie in `[0, 1]`, though QED alone does not establish suitability for a research goal.

In [ ]:
sample = pd.DataFrame(
    {
        "name": [
            "ethanol",
            "propanol",
            "methoxypropane",
            "methoxyethane",
            "ethylamine",
            "propylamine",
            "acetic acid",
            "ethyl acetate",
        ],
        "smiles": [
            "CCO",
            "CCCO",
            "CCCOC",
            "CCOC",
            "CCN",
            "CCCN",
            "CC(=O)O",
            "CCOC(=O)C",
        ],
    }
)
input_mols = [get_mol(s) for s in sample["smiles"]]
assert all(m is not None for m in input_mols), "Input contains an invalid SMILES."
for mol in input_mols:
    validate_reassembly_mol(mol)
sample["qed"] = [QED.qed(m) for m in input_mols]
sample.to_csv(OUTPUT / "input_molecules.csv", index=False)
display(sample)
display(
    Draw.MolsToGridImage(
        input_mols,
        legends=sample["name"].tolist(),
        molsPerRow=4,
        subImgSize=(220, 160),
        useSVG=True,
    )
)

## 3. Mine Building Blocks with `fast-gspan`

Substructures are mined with the `fast-gspan` C++ backend (installed by `uv sync`).
`raw` treats each molecule as an atom-bond graph. Here we search for **substructures of 2–4 atoms that appear in at least 2 molecules**.
`support` is the minimum number of supporting molecules, not a fraction. For `jt` (junction-tree) mining that accounts for rings and cliques, see [decomposition.ipynb](decomposition.ipynb).

The mined molecules are validated, deduplicated, and written to `building_blocks.csv`.
You can pass this file directly as `fragments.path` to the shared factory and CLI.
The loader expects an index column first, followed by the SMILES column.

In [ ]:
from moldr.core.decompose import DefaultConfig, MolsMining

mining_config = DefaultConfig(
    data_path=OUTPUT / "toy.data",
    support=2,
    lower=2,
    upper=4,
    method="raw",
    is_save=False,
    backend="fast-gspan",
    num_threads=2,
)
miner = MolsMining(mining_config)
gspan = miner.decompose(input_mols)
mined_mols = miner.gspan_to_mols(gspan, smiles_list=sample["smiles"].tolist())

fragment_smiles = set()
for mol in mined_mols:
    if mol is None:
        continue
    canonical = Chem.MolToSmiles(mol)
    parsed = get_mol(canonical)
    if parsed is None:
        continue
    validate_reassembly_mol(parsed)
    fragment_smiles.add(get_smiles(parsed))
fragment_smiles = sorted(fragment_smiles)
assert fragment_smiles, "No substructures mined; check support and atom count bounds."
fragments = pd.DataFrame({"smiles": fragment_smiles})
fragment_path = OUTPUT / "building_blocks.csv"
fragments.to_csv(fragment_path)  # The loader expects an index column.
print(f"Input: {len(input_mols)} molecules -> building blocks: {len(fragments)}")
display(fragments)
display(
    Draw.MolsToGridImage(
        [get_mol(s) for s in fragment_smiles],
        legends=fragment_smiles,
        molsPerRow=5,
        subImgSize=(180, 140),
        useSVG=True,
    )
)

With this sample and configuration, 13 building blocks are extracted (such as `CC`, `CO`, and `CCO`).
Changing the input molecules, `support`, or atom-count bounds changes both the count and the extracted fragments.
Before moving to a larger dataset, inspect the grid image above to verify that the mined substructures look reasonable.

## 4. Inspect Reassembly Candidates

Next, we reassemble the mined fragment `CC(=O)O` onto `CCO`. Because reassembly enumerates ways to share atoms or ring bonds between the two structures, it is not mere SMILES concatenation.

We generate candidates with `CandidateGenerator`—used by both `legacy` and `E2E`—and compare their QED scores and structures.
This cell scores all candidates for inspection; during a normal `E2E` transition, only the chosen candidate is scored.

In [ ]:
from moldr.chemistry import CandidateGenerator

base_smiles = "CCO"
attach_smiles = "CC(=O)O"
assert attach_smiles in fragment_smiles
with rdBase.BlockLogs():  # Invalid intermediate merges are discarded by the generator.
    candidates = CandidateGenerator().generate(
        get_mol(base_smiles), get_mol(attach_smiles)
    )
assert candidates, "No valid reassembly candidates were generated."
candidate_table = pd.DataFrame(
    {
        "candidate": range(len(candidates)),
        "smiles": [c.smiles for c in candidates],
        "qed": [QED.qed(c.mol) for c in candidates],
    }
)
candidate_table.to_csv(OUTPUT / "reassembly_candidates.csv", index=False)
display(candidate_table)
display(
    Draw.MolsToGridImage(
        [c.mol for c in candidates[:12]],
        legends=[
            f"{i}: QED={q:.3f}" for i, q in enumerate(candidate_table["qed"][:12])
        ],
        molsPerRow=4,
        subImgSize=(220, 170),
        useSVG=True,
    )
)

## 5. Run One Episode in the Legacy Environment

In `legacy` mode, a single step executes: **agent selects a building block -> environment scores every reassembly candidate and picks the highest-scoring one**. Observations are 300-dimensional vectors from the fixed Mol2Vec model.
By contrast, the **experimental** `E2E` mode chooses the building block and the reassembly candidate as two separate actions.

Here we sample building-block indices uniformly at random without a trained model and use QED to rank reassembly candidates. This is a mechanics check of a tiny environment rather than an optimization benchmark.

Episodes stop after at most 8 steps (at most 8 reassemblies). Matching `legacy` defaults, we enable `sparse` rewards and score-decrease termination. Setting `score_threshold=1.1` keeps QED in `[0, 1]` from ending the episode early by threshold.
Episodes terminate when no candidates remain, the score decreases, or the atom count exceeds `atom_limit` (`atom_count > atom_limit`, so the final molecule can exceed `atom_limit`).

In [ ]:
from moldr.experiment import load_config
from moldr.factory import make_environment, resolve_config

config = load_config(
    cli_values={
        "environment.mode": "legacy",
        "environment.base_smiles": "C",
        "environment.atom_limit": 12,
        "environment.max_episode_steps": 8,
        "environment.score_threshold": 1.1,
        "environment.reward": "sparse",
        "environment.terminate_on_score_decrease": True,
        "fragments.path": str(fragment_path),
        "representation.model_path": str(model_path),
        "scoring.objective": "QED",
        "training.seed": SEED,
        "training.iterations": 1,
        "training.num_workers": 0,
        "training.num_cpus": 2,
        "training.num_gpus": 0.0,
        "training.train_batch_size": 64,
        "training.minibatch_size": 32,
        "training.num_epochs": 1,
        "training.hidden": 32,
        "evaluation.episodes": 3,
        "output.directory": str(OUTPUT / "ppo"),
        "output.logger": "local",
    }
)
config, loaded_blocks, _ = resolve_config(config)
assert loaded_blocks == fragment_smiles
config_path = OUTPUT / "tutorial.yaml"
config_path.write_text(
    yaml.safe_dump(config.to_dict(), sort_keys=False), encoding="utf-8"
)
env = make_environment(config)
obs, info = env.reset(seed=SEED)
assert env.observation_space.contains(obs)
assert env.action_space.n == len(loaded_blocks)
print("Environment:", type(env).__name__, "/ mode:", config.environment.mode)
print("Observation shape / dtype:", obs.shape, obs.dtype)
print("Building-block actions:", env.action_space.n)
print("Initial SMILES / QED:", env.base_smiles, round(env.prev_reward, 4))

Each action is an index into the building-block table. The next cell chooses indices at random and records the reassembly candidate with the highest QED. If no valid candidates exist, the episode ends on the current molecule.

In the table below, `selected_block` is the chosen building block, `candidate_count` is the number of generated candidates, and `smiles` / `qed` are the resulting molecule and score.
With `sparse` rewards, steps return `0` until an environment termination condition is met, at which point `final_weight * QED` is returned (the default `final_weight` is `1`).
`terminated` indicates an environment termination condition, while `truncated` indicates reaching the step limit (truncation alone adds no completion bonus).

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
terminated = truncated = False
try:
    for decision in range(1, config.environment.max_episode_steps + 1):
        action = int(rng.integers(env.action_space.n))
        with rdBase.BlockLogs():
            obs, reward, terminated, truncated, info = env.step(action)
        assert env.observation_space.contains(obs)
        rows.append(
            {
                "decision": decision,
                "action": action,
                "selected_block": loaded_blocks[action],
                "candidate_count": len(info["gen_smiles"]),
                "smiles": env.base_smiles,
                "qed": info["score"],
                "reward": reward,
                "terminated": terminated,
                "truncated": truncated,
            }
        )
        if terminated or truncated:
            break
    assert terminated or truncated, "Episode did not finish within max_episode_steps."
finally:
    env.close()
trajectory = pd.DataFrame(rows)
display(trajectory)

## 6. Save and Inspect the Results

`trajectory.csv` records every decision step, and `episode.json` records the final molecule and termination status.
In the molecule grid below, each unique SMILES is shown once. Even with a fixed random seed, trajectories can change if library or candidate-generation changes affect candidate ordering.

In [ ]:
trajectory.to_csv(OUTPUT / "trajectory.csv", index=False)
summary = {
    "policy": "uniform_building_blocks",
    "seed": SEED,
    "mode": config.environment.mode,
    "candidate_selection": "greedy_score",
    "smiles": rows[-1]["smiles"],
    "qed": float(rows[-1]["qed"]),
    "total_reward": float(trajectory["reward"].sum()),
    "decisions": len(rows),
    "terminated": bool(terminated),
    "truncated": bool(truncated),
}
(OUTPUT / "episode.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
assert get_mol(summary["smiles"]) is not None
unique_states = trajectory.drop_duplicates("smiles")
display(
    Draw.MolsToGridImage(
        [get_mol(s) for s in unique_states["smiles"]],
        legends=[
            f"step {r.decision}, QED={r.qed:.3f}" for r in unique_states.itertuples()
        ],
        molsPerRow=4,
        subImgSize=(220, 170),
        useSVG=True,
    )
)
print(json.dumps(summary, indent=2))
print(
    "Saved files:", ", ".join(p.name for p in sorted(OUTPUT.iterdir()) if p.is_file())
)

## 7. Optional: Run Short CPU PPO Training with the Same Config

This trains a small PPO policy using the exact same `legacy` environment and `sparse` reward settings. Everything beyond the dry-run executes only when `RUN_TRAINING = True`.
First, CLI `--dry-run` validates the configuration and assets; this check runs during **Run All** as well.

Training uses a minimal configuration: 1 iteration, batch size 64, 1 SGD epoch, 0 rollout workers, and 0 GPUs.
Even with `num_workers=0`, **local Ray must still start**, so run this in an environment that allows local processes and sockets. It is kept opt-in so the initial notebook smoke check stays quick.

This 1-iteration run checks training, saving, and restoration wiring; it is not expected to outperform random selection.
`cli.main([...])` uses the same entry point as running `python -m moldr.cli ...` in a terminal.

In [ ]:
from moldr.cli import main as moldr_cli

RUN_TRAINING = False
train_args = ["train", "--config", str(config_path)]
with redirect_stdout(io.StringIO()):
    checked_config = moldr_cli([*train_args, "--dry-run"])
assert checked_config.environment.mode == "legacy"
assert checked_config.environment.reward == "sparse"
print(
    "CLI dry-run: OK; CPU / legacy / sparse / QED /",
    len(loaded_blocks),
    "building blocks",
)

training_run = None
if RUN_TRAINING:
    training_run = moldr_cli(train_args)
    print("Training run:", training_run)
else:
    print(
        "Training skipped. Set RUN_TRAINING = True and re-run from this cell to try it."
    )

## 8. Optional: Restore Checkpoint and Sample Molecules

If you ran the training cell above, this cell generates 3 episodes using the `legacy` configuration and assets saved in that run directory.
You can pass the run directory directly to `--checkpoint`.
Restore from the saved configuration rather than manually altering the observation dimension or fragment table.

In [ ]:
if training_run is not None:
    evaluation_run = moldr_cli(
        [
            "sample",
            "--checkpoint",
            str(training_run),
            "--episodes",
            "3",
        ]
    )
    generated = pd.read_json(evaluation_run / "molecules.json")
    display(generated[["seed", "smiles", "score", "steps", "terminated", "truncated"]])
    display(
        Draw.MolsToGridImage(
            [get_mol(s) for s in generated["smiles"]],
            legends=[f"QED={s:.3f}" for s in generated["score"]],
            molsPerRow=3,
            subImgSize=(220, 170),
            useSVG=True,
        )
    )
    print("Sampling output:", evaluation_run)
else:
    print("Run the optional training cell above first to sample from a checkpoint.")

## Where to Substitute Your Own Research Data

| What to change | Entry point | What to verify |
| --- | --- | --- |
| Input molecules | `sample` in Section 2 | Every SMILES parses cleanly; start with a small batch |
| Mining granularity | `support`, `lower`, `upper`, `method` in Section 3 | Number and chemistry of mined frequent substructures |
| Existing building-block table | `fragments.path` in Section 5 | CSV with leading index column + SMILES in the next column; `.smi` / `.txt` also allows one SMILES per line |
| Score | `scoring.objective` | Built-in `QED`, `qed-sa`, `logp-8`, `logp-minus1` (update `qed` labels in tables if changed) |
| Custom objective | `moldr.scoring.register_objective` | Higher values represent better candidates; register before constructing environments or training |
| Initial molecule / search bounds | `environment.base_smiles`, `atom_limit`, `max_episode_steps` | Compute cost and termination behavior (`legacy` scores all candidates every step) |
| Reward / early stopping | `environment.reward`, `final_weight`, `terminate_on_score_decrease` | Default `legacy` uses `sparse` + score-decrease termination; `improvement` adds step deltas |
| Training budget | `training.iterations` and PPO batch / epoch settings | Compare across multiple seeds, independent evaluation settings, and a random baseline |

To try a bundled mined building-block table, point `fragments.path` to e.g. `ROOT / "data/building_blocks/guacamol/guacamol_s10000_l2.csv"`. Because it is much larger than the 13-fragment toy table, compute cost increases accordingly.

The current reassembly implementation cannot handle explicit atomic or bond stereochemistry, and the environments reject inputs containing it. Additionally, fixes such as preserving formal charges and isotopes can alter generated molecules compared to earlier versions; agreement on neutral achiral tests should be verified separately from full input and checkpoint compatibility.

For more details, see [experiment configuration](../docs/experiment_configuration.md) and [faster reassembly and experimental extensions](../README.md#faster-reassembly-and-experimental-extensions).